# חיים קל: אימון הקול הקל (Piper) ב-Colab

לפני ההרצה: להעלות ל-Google Drive את `dataset.zip` (מ-`make_dataset_zip.py`) לתיקייה `otzaria-light/`.

סדר ההרצה: תא אחרי תא. אם הסשן מתנתק, מריצים שוב מההתחלה: האימון ממשיך מנקודת השמירה האחרונה ב-Drive.

Runtime ← Change runtime type ← GPU (T4 מספיק; L4/A100 מהירים יותר).

In [ ]:
# 1. Drive and folders
from google.colab import drive
drive.mount('/content/drive')
import os
ROOT = '/content/drive/MyDrive/otzaria-light'
RUN = 'pilot'            # a name for this training run
BASE = 'he_IL-saspeech'  # 'he_IL-saspeech' (pilot only, non-commercial) or 'en_US-lessac' (free)
OUT = f'{ROOT}/runs/{RUN}'
os.makedirs(OUT + '/ckpt', exist_ok=True)
assert os.path.exists(f'{ROOT}/dataset.zip'), 'upload dataset.zip to MyDrive/otzaria-light first'
!nvidia-smi -L

In [ ]:
# 2. Piper's training code (GPL-3; only the training runs here, the voice is ours)
%cd /content
!apt-get -qq install -y build-essential cmake ninja-build > /dev/null
!git clone -q https://github.com/OHF-Voice/piper1-gpl.git
%cd /content/piper1-gpl
!pip install -q -e '.[train]'
!./build_monotonic_align.sh
!python3 -m piper.train fit --help | head -40

In [ ]:
# 3. The data
!rm -rf /content/data && mkdir -p /content/data
!unzip -q {ROOT}/dataset.zip -d /content/data
!wc -l /content/data/metadata.csv && ls /content/data/wavs | wc -l
!head -3 /content/data/metadata.csv

In [ ]:
# 4. The base checkpoint (medium, 22.05 kHz) and our symbol table
from huggingface_hub import list_repo_files, hf_hub_download
folder = {'he_IL-saspeech': 'he/he_IL/saspeech/medium', 'en_US-lessac': 'en/en_US/lessac/medium'}[BASE]
files = [f for f in list_repo_files('rhasspy/piper-checkpoints', repo_type='dataset') if f.startswith(folder + '/') and f.endswith('.ckpt')]
print(files)
base_ckpt = hf_hub_download('rhasspy/piper-checkpoints', files[0], repo_type='dataset', local_dir='/content/base')
print(base_ckpt)

# Piper's default symbol table has no Hebrew, so the voice gets its own: every character of the
# vocalized text (NFD) is one symbol. 256 slots like the base model, so the embedding shape matches.
# The table is written once per run and reused on a resume, so ids never move mid-training.
import csv, json, unicodedata
PHONEMES = f'{OUT}/phonemes.json'
if os.path.exists(PHONEMES):
    id_map = json.load(open(PHONEMES, encoding='utf-8'))
else:
    chars = set()
    with open('/content/data/metadata.csv', encoding='utf-8') as f:
        for row in csv.reader(f, delimiter='|'):
            chars.update(unicodedata.normalize('NFD', row[-1]))
    chars -= {'_', '^', '$'}
    id_map = {'_': [0], '^': [1], '$': [2]}
    for i, ch in enumerate(sorted(chars), start=3):
        id_map[ch] = [i]
    assert len(id_map) <= 256, len(id_map)
    json.dump(id_map, open(PHONEMES, 'w', encoding='utf-8'), ensure_ascii=False)
print(len(id_map), 'symbols:', ''.join(k for k in id_map if k not in '_^$'))


In [ ]:
# 5. Training. First run: warm start from the base checkpoint (every weight whose shape matches is
#    copied, the rest starts fresh). Later runs resume from this run's newest checkpoint in Drive.
#    Expect a line "[warmstart] Copied N parameters" with N in the hundreds, then one line per epoch.
import glob, os
ckpts = sorted(glob.glob(f'{OUT}/ckpt/*.ckpt'), key=os.path.getmtime)
EPOCHS = 200 if RUN == 'pilot' else 1500     # ~85 steps per epoch (1,400 clips, batch 16); pilot ~3-4 h on a T4
start = f'--ckpt_path "{ckpts[-1]}"' if ckpts else f'--model.warmstart_ckpt "{base_ckpt}"'
print('starting with', start)
!python3 -m piper.train fit \
  --data.voice_name chaim-light \
  --data.csv_path /content/data/metadata.csv \
  --data.audio_dir /content/data/wavs \
  --data.phoneme_type text \
  --data.espeak_voice he \
  --data.phonemes_path {PHONEMES} \
  --model.sample_rate 22050 \
  --data.cache_dir /content/cache \
  --data.config_path {OUT}/chaim-light.json \
  --data.batch_size 16 \
  --data.num_workers 2 \
  --trainer.max_epochs {EPOCHS} \
  --trainer.default_root_dir /content/runs \
  --trainer.callbacks+=ModelCheckpoint \
  --trainer.callbacks.dirpath {OUT}/ckpt \
  --trainer.callbacks.every_n_train_steps 500 \
  --trainer.callbacks.save_top_k 1 \
  {start}


In [ ]:
# 6. Export to ONNX (what the service loads)
import glob, os
last = sorted(glob.glob(f'{OUT}/ckpt/*.ckpt'), key=os.path.getmtime)[-1]
!python3 -m piper.train.export_onnx --checkpoint "{last}" --output-file {OUT}/chaim-light.onnx
!ls -la {OUT}

In [ ]:
# 7. Listen: the test sentences through the new voice
import json, wave, unicodedata
import numpy as np, onnxruntime as ort
from IPython.display import Audio, display
cfg = json.load(open(f'{OUT}/chaim-light.json', encoding='utf-8'))
id_map = cfg['phoneme_id_map']
sess = ort.InferenceSession(f'{OUT}/chaim-light.onnx')
def say(text):
    text = unicodedata.normalize('NFD', text)
    ids = list(id_map['^']) + list(id_map['_'])   # Piper: bos, pad, (symbol, pad)*, eos
    for ch in text:
        if ch in id_map:
            ids += id_map[ch] + id_map.get('_', [0])
    ids += id_map.get('$', [])
    x = np.array([ids], dtype=np.int64)
    out = sess.run(None, {'input': x, 'input_lengths': np.array([x.shape[1]], dtype=np.int64),
                          'scales': np.array([0.667, 1.0, 0.8], dtype=np.float32)})[0]
    return out.squeeze()
os.makedirs(f'{OUT}/samples', exist_ok=True)
for i, line in enumerate(open('/content/data/test_sentences.txt', encoding='utf-8')):
    pcm = say(line.strip())
    with wave.open(f'{OUT}/samples/{i:02d}.wav', 'wb') as w:
        w.setnchannels(1); w.setsampwidth(2); w.setframerate(cfg['audio']['sample_rate'])
        w.writeframes((np.clip(pcm, -1, 1) * 32767).astype(np.int16).tobytes())
    print(line.strip()); display(Audio(pcm, rate=cfg['audio']['sample_rate']))
print('samples saved to', f'{OUT}/samples')